In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.gold-helpers

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_drivers"

In [0]:
drivers_df = (
    spark.table(f"{catalog_name}.{silver_schema}.drivers")
         .filter(F.col("batch_id") == v_batch_id)
)

In [0]:
ref_nationality_region_df = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

In [0]:
dim_drivers_df = (
    drivers_df.alias("dr")
        .join(
            ref_nationality_region_df.alias("nrg")
            , F.col("dr.nationality") == F.col("nrg.nationality")
            , "left"
        )
        .selectExpr(
            "dr.driver_id"
            , "dr.driver_name"
            , "dr.date_of_birth"
            , "dr.nationality"
            , "nrg.region as nationality_region"
        )
)
display(dim_drivers_df)

In [0]:
write_to_gold(
    input_df=dim_drivers_df,
    target_table=target_table,
    merge_condition="t.driver_id = s.driver_id",
    columns_to_update=[
        "driver_name",
        "date_of_birth",
        "nationality",
        "nationality_region"
    ]
)

In [0]:
display(spark.table(target_table))